# Housing Price Prediction

Dataset Description: The data pertains to the houses found in each California district and some summary statistics about them based on the 1990 census data. It contains one instance per district block group. A block group is the smallest geographical unit for which the U.S. Census Bureau publishes sample data (a block group typically has a population of 600 to 3,000 people). 

The goal of this task is to design a regression model to predict the median house value conditioned upon a set of input attributes corresponding to a particular California district block. 

The attributes in the dataset are as follows; their names are self-explanatory: 
     

    longitude (continuous): One of the coordinates that are used to identify the California district block 
     

    latitude (continuous): One of the coordinates that are used to identify the California district block 
     

    housing_median_age (continuous): Average age of the house in California district block 
     

    total_rooms (continuous): Total number of rooms of all the houses in the California district block 
     

    total_bedrooms (continuous): Total number of bedrooms of all the houses in the California district block 
     

    population (continuous): Number of people residing in the district block 
     

    households (continuous): Number of families in the district block 
     

    median_income (continuous): Median income for households in the district block of houses (measured in tens of thousands of US Dollars)  
     

    ocean_proximity (categorical): Location of the house. Is it inland, near the bay, near the ocean, etc.  
     

    median_house_value.(continuous): Median house value within a district block (measured in US Dollars)

Our target variable will be median_house_value.  Use the rest of the fields mentioned above to predict the median_house_value. 

### a) Import Libraries
Import all necessary libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score

### b. Data Loading / Preprocessing

#### i. Loading

1. Load the California housing dataset using `pandas.read_csv()` function and store it in the variable (i.e., a pandas dataframe) named `df’.

2. The resulting data frame should have the shape (20640, 10) indicating that there are 20640 rows and 10 columns.

In [ ]:
df = pd.read_csv('/kaggle/input/house-price/1553768847-housing.csv')
df.head(5)

In [ ]:
df.shape

3. Find the missing values in the data frame. If any (i.e., even if one column in each instance / row has a missing value), drop the row using `pandas.DataFrame.dropna()` function. The resulting data frame should have the shape (20433, 10) indicating that there are 20433 rows and 10 columns.

In [ ]:
df = df.dropna()
print(df.shape)

4. Create a data frame `corr_df` by dropping the columns latitude, longitude, and ocean_proximity using the `pandas.DataFrame.drop()` function. Use the Pearson correlation to find the correlation of each remaining feature in the `corr_df` with the target variable `median_house_value` using the function `pandas.DataFrame.corrwith()`. 

In [ ]:
df_filltered = df.drop(columns=['latitude','longitude','ocean_proximity'])
corr_df = df_filltered.corrwith(df_filltered['median_house_value'],method='pearson')
corr_df

5. Create a data frame `X` of features (by dropping the column `median_house_value` from the original data frame) using the `pandas.DataFrame.drop()` function. Create a Series object of targets `Y` (by only considering the `median_house_value` column from the original data frame (Do NOT use the `corr_df` data frame in this step. Use the data frame which was obtained as a result of step 3 above). 

In [ ]:
y = df['median_house_value']
X = df.drop(columns=['median_house_value'])
print(X)
print(y)

#### ii. Data Visualization

1. Use `pandas.DataFrame.hist(bins = 50)` function for visualizing the variation on the columns housing_median_age, total_rooms, total_bedrooms, population, household, median_income and median_house_value. Plot each histogram as a separate subplot.

In [ ]:
ax = X.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

2. Use `pandas.dataframe.describe()` function to find the mean, median and standard deviations for each feature and report in the jupyter notebook.

In [ ]:
X.describe()

3. Use `pandas.get_dummies` to convert categorical variables into dummy /one-hot encoding. In this case the categorical column is ocean_proximity 

In [ ]:
dummies = pd.get_dummies(X.ocean_proximity, dtype=float)

In [ ]:
dummies.head(5)

In [ ]:
X = X.drop(columns=['ocean_proximity'], axis=1)
X = pd.concat([X, dummies], axis=1)
print(X)

#### iii. Data Splitting

1. Split data into training and test sets using the sklearn `train_test_split()` function. Perform 70-30 distribution i.e. 70% training and 30% testing. The result of your data split should yield 4 separate data frames `X_train, X_test, y_train, y_test`. (respectively, the training features, testing features, training targets and testing target).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(y_train)

#### iv. Data Scaling

1. Use the `StandardScaler()` to instantiate the standard scaler class. Note: You will need two separate scaler objects, one to scale the features, another to scale the target values. 

In [ ]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

2. For each scaler, employ the `fit_transform()` function (only on the training  features, training targets) of the scaler to retrieve the new (scaled) version of the data. Store them in `X_train`, and `y_train` again

In [ ]:
X_train = x_scaler.fit_transform(X_train)
y_train = y_scaler.fit_transform(np.array(y_train).reshape(-1,1))

3. Scale the `X_test` and `y_test` as well and store the scaled values back in `X_test` and `y_test`. (i.e., use the appropriate “fitted” scaler above to “transform” the test data. Note: the function to be employed in this case is `transform()` as opposed to `fit_transform()`).  
Henceforth, `X_train, y_train, X_test, y_test` will refer to the scaled data unless stated otherwise.

In [ ]:
X_test = x_scaler.transform(X_test)
y_test = y_scaler.transform(np.array(y_test).reshape(-1,1))

4. Use `pandas.DataFrame.hist(bins = 50)` function for visualizing the variation of numerical attributes housing_median_age, total_rooms, total_bedrooms, population, household, median_income and median_house_value for the `X_train` and `y_train` dataset (similar to step b.ii.1 above). Once again, plot each histogram as a separate subplot. 

In [ ]:
ax = X.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

### c. Modelling

#### i. Employ Linear Regression from sklearn.linear_model, and instantiate the model.

In [ ]:
reg = LinearRegression()

#### ii. Once instantiated, `fit()` the model using the scaled `X_train, y_train` data.

In [ ]:
reg.fit(X_train, y_train)

#### iii. Employ the `predict()` function to obtain predictions on `X_test`. Store the predictions in a variable named `y_preds`. Note: Since the model has been trained on scaled data (i.e., both features and targets, the predictions will also be in the “scaled” space. We need to transform the predictions back to the original space). 

In [ ]:
y_preds = reg.predict(X_test)

#### iv. Use `inverse_transform()` function to convert the normalized data (`y_preds` ) to original scale. Store the transformed values back into `y_preds`.

In [ ]:
y_preds = y_scaler.inverse_transform(y_preds)
y_test = y_scaler.inverse_transform(y_test)
print(y_preds,y_test)

#### v. Perform PCA on the features (`X_train`) and set `n_component` as 2.

In [ ]:
pca = PCA(n_components=2)
pca.fit_transform(X_train)

1. Show a scatter plot where on the x-axis we plot the first PCA component and second component on the y-axis.

In [ ]:
Xt = pca.fit_transform(X_train)
plot = plt.scatter( Xt[:,0], Xt[:,1], c=y_train)
plt.xlabel("PCA-1")
plt.ylabel("PCA-2")
plt.show()

2. Calculate the total percentage of variance captured by the 2 PCA components using `pca.explained_variance_ratio_`. Also, report the strength of each PCA component using `pca.singular_values_`.

In [ ]:
print("Percentage of variance captured: ",pca.explained_variance_ratio_*100)
print("Strength of each PCA components: ",pca.singular_values_)

### d. Evaluation

#### i. Plot a scatter plot using matplotlib.pyplot.scatter function. Plot the predicted median house values on the y-axis vs the actual median house values on the x-axis

In [ ]:
plt.scatter(y_test, y_preds, edgecolor='orange', alpha=0.5, color='blue')
plt.xlabel("Actual House Price")
plt.ylabel("Predicated House Price")
plt.grid()

#### ii. Calculate MAPE, RMSE and R2 for the model and report them in the following table.  
Hint: for RMSE set the squared parameter to False.

In [ ]:
mape = mean_absolute_percentage_error(y_test, y_preds)
rmse = mean_squared_error(y_test, y_preds, squared=False)
r2 = r2_score(y_test, y_preds)

print(f"MAPE : {mape}")
print(f"RMSE : {rmse}")
print(f"R2 : {r2}")

# Discussion Board Calculation

In [ ]:
coeff = reg.coef_
coeff_dict = {}

feature_names = X.columns
for i, feature_name in enumerate(feature_names):
    coeff_dict[feature_name] = coeff[0][i]

coeff_dict = sorted(coeff_dict.items(), key=lambda item: item[1], reverse=True)

for i in coeff_dict:
    print(i)

In [ ]:
corr_df